<a href="https://colab.research.google.com/github/littlehorsebrother2021/python_code_files/blob/main/yolov2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 2. 创建一个统一的原始数据存放目录
!mkdir -p /content/wider_face_raw

# ================= 核心修改区域 =================
# 请分别把下面三个双引号里的路径，替换为你网盘中对应的真实快捷方式路径！

# ① 映射训练集压缩包 (WIDER_train.zip)
!ln -s "/content/drive/MyDrive/WIDER_train.zip" /content/wider_face_raw/WIDER_train.zip

# ② 映射验证集压缩包 (WIDER_val.zip)
!ln -s "/content/drive/MyDrive/WIDER_val.zip" /content/wider_face_raw/WIDER_val.zip

# ③ 映射标注文件解压后的文件夹或压缩包（包含 wider_face_train_bbx_gt.txt 的那个）
!ln -s "/content/drive/MyDrive/wider_face_split.zip" /content/wider_face_raw/wider_face_split.zip
# ===============================================

# 3. 创建本地极速 SSD 目录并解压
!mkdir -p /content/wider_face_local
!unzip -q /content/wider_face_raw/WIDER_train.zip -d /content/wider_face_local/
!unzip -q /content/wider_face_raw/WIDER_val.zip -d /content/wider_face_local/
!unzip -q /content/wider_face_raw/wider_face_split.zip -d /content/wider_face_local/

# 4. 验证文件是否全部就位
print("\n--- 检查本地虚拟盘中的文件布局 ---")
!ls -l /content/wider_face_local/
print("\n--- 检查标注文件是否存在 ---")
!ls -l /content/wider_face_raw/wider_face_split/

ln: failed to create symbolic link '/content/wider_face_raw/WIDER_train.zip': File exists
ln: failed to create symbolic link '/content/wider_face_raw/WIDER_val.zip': File exists
ln: failed to create symbolic link '/content/wider_face_raw/wider_face_split.zip': File exists
replace /content/wider_face_local/WIDER_train/images/0--Parade/0_Parade_marchingband_1_100.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/wider_face_local/WIDER_train/images/0--Parade/0_Parade_marchingband_1_1015.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 
error:  invalid response [{ENTER}]
replace /content/wider_face_local/WIDER_train/images/0--Parade/0_Parade_marchingband_1_1015.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/wider_face_local/WIDER_train/images/0--Parade/0_Parade_marchingband_1_1018.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/wider_face_local/WIDER_train/images/0--Parade/0_Parade_marchingband_1_1022.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /

下面对标签进行格式转换

In [ ]:
import os
import cv2

def convert_wider_to_yolo(txt_path, img_dir, out_label_dir, out_list_txt, is_test=False):
    """
    Wider Face 标注格式转标准 YOLO 格式脚本
    :param txt_path: 官方标注文本路径（如 wider_face_train_bbx_gt.txt）
    :param img_dir: 对应图片的根目录（如 WIDER_train/images）
    :param out_label_dir: 转换后的 YOLO .txt 标签存放目录
    :param out_list_txt: 生成的路径清单文件（如 face_train.txt），供开源库训练读取
    :param is_test: 是否为测试集（测试集没有公开标注，只需生成路径清单）
    """
    os.makedirs(out_label_dir, exist_ok=True)
    total_images = 0

    with open(out_list_txt, 'w') as list_f:
        # 如果是测试集，没有官方的标注文本，我们直接遍历图片生成路径清单即可
        if is_test or not txt_path or not os.path.exists(txt_path):
            print(f"开始处理测试集图片清单...")
            for root, dirs, files in os.walk(img_dir):
                for file in files:
                    if file.endswith('.jpg') or file.endswith('.png'):
                        # 写入绝对路径
                        list_f.write(os.path.join(root, file) + '\n')
                        total_images += 1
            print(f"测试集处理完成，共计 {total_images} 张图片。")
            return

        print(f"开始解析标注文件: {txt_path}")
        with open(txt_path, 'r') as f:
            lines = f.readlines()

        i = 0
        while i < len(lines):
            line = lines[i].strip()
            # 识别到图片路径行（例如：0--Parade/0_Parade_marchingband_1_849.jpg）
            if line.endswith('.jpg') or line.endswith('.png'):
                img_path_rel = line
                img_path_full = os.path.join(img_dir, img_path_rel)

                # 读取图片以获取分辨率（宽 w 和高 h），用于 YOLO 坐标归一化
                img = cv2.imread(img_path_full)
                if img is None:
                    # 如果图片解压损坏或不存在，跳过对应的数据行
                    i += 1
                    if i < len(lines):
                        num_boxes = int(lines[i].strip())
                        i += num_boxes + 1
                    continue
                h, w, _ = img.shape

                # 将该图片的完整路径写入清单文件
                list_f.write(img_path_full + '\n')
                total_images += 1

                # 生成对应的 YOLO .txt 标签文件路径（保持文件名一致，放进 labels 目录）
                img_name = os.path.basename(img_path_rel)
                label_name = os.path.splitext(img_name)[0] + '.txt'
                label_txt_path = os.path.join(out_label_dir, label_name)

                # 下一行是人脸框的数量
                i += 1
                num_boxes = int(lines[i].strip())

                # 循环读取每一个人脸框坐标
                with open(label_txt_path, 'w') as label_f:
                    for _ in range(num_boxes):
                        i += 1
                        box_line = lines[i].strip().split()
                        if len(box_line) < 4:
                            continue

                        # Wider Face 格式: x1, y1, w, h (绝对像素值)
                        x1, y1, box_w, box_h = list(map(int, box_line[:4]))

                        # 过滤无效的或严重模糊不可见的干扰框
                        if box_w <= 0 or box_h <= 0:
                            continue

                        # 核心转换算法：Wider Face (左上角 x1,y1) -> YOLO (中心点 x_center, y_center) 并归一化到 0~1
                        x_center = (x1 + box_w / 2.0) / w
                        y_center = (y1 + box_h / 2.0) / h
                        norm_w = box_w / w
                        norm_h = box_h / h

                        # 写入标签，因为只有“人脸”一类，所以类别 ID 固定为 0
                        label_f.write(f"0 {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}\n")
            i += 1

    print(f"成功处理完该数据集，转换后共计 {total_images} 张有效图片。")

# ==================== 开始执行转换 ====================
print("--- 1. 开始转换训练集 (Train) ---")
convert_wider_to_yolo(
    txt_path='/content/wider_face_local/wider_face_split/wider_face_train_bbx_gt.txt',
    img_dir='/content/wider_face_local/WIDER_train/images',
    out_label_dir='/content/wider_face_local/WIDER_train/labels',
    out_list_txt='face_train.txt'
)

print("\n--- 2. 开始转换验证集 (Val) ---")
convert_wider_to_yolo(
    txt_path='/content/wider_face_local/wider_face_split/wider_face_val_bbx_gt.txt',
    img_dir='/content/wider_face_local/WIDER_val/images',
    out_label_dir='/content/wider_face_local/WIDER_val/labels',
    out_list_txt='face_val.txt'
)

print("\n--- 3. 开始生成测试集清单 (Test) ---")
# 官方测试集无公开标注，因此 txt_path 传 None，脚本将只批量搜集图片路径用于后续推理评估
convert_wider_to_yolo(
    txt_path=None,
    img_dir='/content/wider_face_local/WIDER_test/images',
    out_label_dir='/content/wider_face_local/WIDER_test/labels',
    out_list_txt='face_test.txt',
    is_test=True
)

print("\n【大功告成】所有格式转换已完成！可以直接进行下一阶段修改 .cfg 文件的操作。")

--- 1. 开始转换训练集 (Train) ---
开始解析标注文件: /content/wider_face_local/wider_face_split/wider_face_train_bbx_gt.txt
成功处理完该数据集，转换后共计 12880 张有效图片。

--- 2. 开始转换验证集 (Val) ---
开始解析标注文件: /content/wider_face_local/wider_face_split/wider_face_val_bbx_gt.txt
成功处理完该数据集，转换后共计 3226 张有效图片。

--- 3. 开始生成测试集清单 (Test) ---
开始处理测试集图片清单...
测试集处理完成，共计 0 张图片。

【大功告成】所有格式转换已完成！可以直接进行下一阶段修改 .cfg 文件的操作。


准备开始训练模型

In [ ]:
# --- 安装与环境配置 (一次性执行) ---
import os

# 1. 克隆官方仓库并编译
if not os.path.exists('/content/darknet'):
    !git clone https://github.com/AlexeyAB/darknet.git
    %cd darknet

    # 开启所有必需选项：GPU、CUDNN、OPENCV、TF（核心！）
    !sed -i 's/GPU=0/GPU=1/g' Makefile
    !sed -i 's/CUDNN=0/CUDNN=1/g' Makefile
    !sed -i 's/OPENCV=0/OPENCV=1/g' Makefile
    !sed -i 's/TF=0/TF=1/g' Makefile  # 开启TensorFlow导出支持
    !make   # 重新编译
    %cd /content/darknet


%cd /content/darknet



/content/darknet


In [ ]:
!pip install onnx onnxscript

  Using cached onnx-1.21.0-cp312-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.5 kB)
Using cached onnx-1.21.0-cp312-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (17.6 MB)
  Attempting uninstall: onnx
    Found existing installation: onnx 1.16.0
    Uninstalling onnx-1.16.0:
      Successfully uninstalled onnx-1.16.0


In [ ]:
import os
import time
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from cv2 import imread, cvtColor, COLOR_BGR2RGB, resize
import numpy as np

# ==============================================================================
# 1. 严格适配 STM32N6 NPU 的模型架构 (Tiny YOLO Face)
# 激活函数全部采用 ReLU / LeakyReLU，输入输出采用静态全尺寸张量
# ==============================================================================

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super(ConvBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = nn.LeakyReLU(0.1, inplace=True) # NPU 硬件完美支持的高效激活函数

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class TinyYoloFaceNet(nn.Module):
    def __init__(self, num_anchors=5, num_classes=1):
        super(TinyYoloFaceNet, self).__init__()
        # 输出通道计算: 5 * (5 + 1) = 30
        self.out_channels = num_anchors * (5 + num_classes)

        # 主干特征提取网络 (Backbone)
        self.layer1 = ConvBlock(3, 16, 3, 1, 1)
        self.pool1 = nn.MaxPool2d(2, 2) # 224x224 -> 112x112

        self.layer2 = ConvBlock(16, 32, 3, 1, 1)
        self.pool2 = nn.MaxPool2d(2, 2) # 112x112 -> 56x56

        self.layer3 = ConvBlock(32, 64, 3, 1, 1)
        self.pool3 = nn.MaxPool2d(2, 2) # 56x56 -> 28x28

        self.layer4 = ConvBlock(64, 128, 3, 1, 1)
        self.pool4 = nn.MaxPool2d(2, 2) # 28x28 -> 14x14

        self.layer5 = ConvBlock(128, 256, 3, 1, 1)
        self.pool5 = nn.MaxPool2d(2, 2) # 14x14 -> 7x7

        self.layer6 = ConvBlock(256, 512, 3, 1, 1)

        # 预测头 (Detection Head) - 输出完全静态的 [Batch, 30, 7, 7] 特征图
        self.conv_head = nn.Conv2d(512, self.out_channels, kernel_size=1, stride=1, padding=0, bias=True)

    def forward(self, x):
        x = self.pool1(self.layer1(x))
        x = self.pool2(self.layer2(x))
        x = self.pool3(self.layer3(x))
        x = self.pool4(self.layer4(x))
        x = self.pool5(self.layer5(x))
        x = self.layer6(x)
        x = self.conv_head(x)
        return x # 极致精简，不包含任何 NMS、Sigmoid 等动态后处理算子

# ==============================================================================
# 2. 数据集加载器 (Dataset & Dataloader)
# ==============================================================================

class YoloFaceDataset(Dataset):
    def __init__(self, list_file, img_size=224, S=7, B=5, C=1):
        with open(list_file, 'r') as f:
            self.lines = [line.strip() for line in f.readlines() if line.strip()]
        self.img_size = img_size
        self.S = S
        self.B = B
        self.C = C

    def __len__(self):
        return len(self.lines)

    def __getitem__(self, idx):
        img_path = self.lines[idx]
        # 根据您转换脚本的逻辑，将 images 替换为 labels 寻找 txt 标注文件
        label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt')

        # 读取并归一化图像 (适配 MCU 端输入规范)
        img = imread(img_path)
        img = cvtColor(img, COLOR_BGR2RGB)
        img = resize(img, (self.img_size, self.img_size))
        img = img.astype(np.float32) / 255.0  # 归一化到 0~1 之间
        img = img.transpose(2, 0, 1)          # 转为 NCHW 排布格式

        # 构建静态目标网格矩阵 [7, 7, 5 + C]
        target = np.zeros((self.S, self.S, 5 + self.C), dtype=np.float32)

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = list(map(float, line.strip().split()))
                    if len(parts) == 5:
                        cls, x, y, w, h = parts
                        # 计算所属的 7x7 网格单元
                        grid_x = int(x * self.S)
                        grid_y = int(y * self.S)
                        if grid_x < self.S and grid_y < self.S:
                            # 仅写入负责该区域的主锚框基础属性
                            target[grid_y, grid_x, 0:5] = [x, y, w, h, 1.0] # 坐标与人脸置信度
                            target[grid_y, grid_x, 5] = 1.0                # 人脸类别 (Class 0)

        return torch.tensor(img, dtype=torch.float32), torch.tensor(target, dtype=torch.float32)

# ==============================================================================
# 3. 极简目标检测损失函数 (Loss Function)
# ==============================================================================

class YoloLoss(nn.Module):
    def __init__(self, S=7, B=5, C=1):
        super(YoloLoss, self).__init__()
        self.S = S
        self.B = B
        self.C = C
        self.mse_loss = nn.MSELoss(reduction='sum')

    def forward(self, pred, target):
        # pred: [Batch, 30, 7, 7] -> 变换为 [Batch, 7, 7, 30]
        pred = pred.permute(0, 2, 3, 1)
        batch_size = pred.size(0)

        # 重塑预测形状以匹配 Anchor 结构
        pred = pred.view(batch_size, self.S, self.S, self.B, 5 + self.C)

        # 提取目标和各掩码
        target_box = target[:, :, :, 0:5].unsqueeze(3).expand_as(pred[:,:,:,:,0:5])
        coord_mask = target[:, :, :, 4].unsqueeze(3).unsqueeze(4) # 有人脸的位置

        # 静态简单损失计算：坐标损失 + 置信度损失
        loss_coord = self.mse_loss(pred[:,:,:,:,0:4] * coord_mask, target_box[:,:,:,:,0:4] * coord_mask)
        loss_conf = self.mse_loss(pred[:,:,:,:,4:5] * coord_mask, target_box[:,:,:,:,4:5] * coord_mask)
        loss_noobj = self.mse_loss(pred[:,:,:,:,4:5] * (1 - coord_mask), target_box[:,:,:,:,4:5] * (0))

        total_loss = (5.0 * loss_coord + loss_conf + 0.5 * loss_noobj) / batch_size
        return total_loss

# ==============================================================================
# 4. 核心训练流水线循环 (Training Pipeline)
# ==============================================================================

if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"当前运行环境: {device}")

    # 载入由 1.txt 生成的训练/验证描述文件
    train_dataset = YoloFaceDataset(list_file='/content/face_train.txt', img_size=224)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)

    model = TinyYoloFaceNet().to(device)
    criterion = YoloLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    print("--- 开始训练任务 ---")
    model.train()
    # 为在 Colab 中快速跑通并验证导出，这里默认设置 3 个 Epoch，您可根据需要调大
    epochs = 3
    for epoch in range(epochs):
        epoch_loss = 0.0
        for imgs, targets in train_loader:
            imgs, targets = imgs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss/len(train_loader):.4f}")

    # 保存训练好的权重
    torch.save(model.state_dict(), 'tiny_yolo_face_weights.pth')
    print("模型权重保存完成。")

    # ==============================================================================
    # 5. 严格遵循 STM32N647 NPU 规格的 ONNX 静态模型导出脚本
    # ==============================================================================
    print("\n--- 开始导出适配 STM32N6 NPU 的优化静态 ONNX ---")
    model.eval()
    model.to('cpu')

    # 融合 BN
    import torch.quantization as quant
    try:
        model = quant.fuse_modules(model, [
            ['layer1.conv', 'layer1.bn', 'layer1.act'],
            ['layer2.conv', 'layer2.bn', 'layer2.act'],
            ['layer3.conv', 'layer3.bn', 'layer3.act'],
            ['layer4.conv', 'layer4.bn', 'layer4.act'],
            ['layer5.conv', 'layer5.bn', 'layer5.act'],
            ['layer6.conv', 'layer6.bn', 'layer6.act']
        ])
        print("BatchNorm 融合成功")
    except:
        print("融合失败，可能是模块名不匹配，跳过（不影响导出）")

    # 导出
    dummy_input = torch.randn(1, 3, 224, 224, device='cpu')
    input_names = ["Input_0"]
    output_names = ["Transpose_55_out_0"]

    torch.onnx.export(
        model,
        dummy_input,
        "tiny_yolo_face_224_static_optimized.onnx",
        opset_version=18,
        do_constant_folding=True,
        input_names=input_names,
        output_names=output_names,
        export_params=True
    )

    # 确保为单文件
    import onnx
    m = onnx.load("tiny_yolo_face_224_static_optimized.onnx")
    onnx.save(m, "tiny_yolo_face_224_static_optimized.onnx")
    print("单文件静态 ONNX 已生成")


当前运行环境: cuda
--- 开始训练任务 ---
Epoch [1/3], Loss: 1.3185
Epoch [2/3], Loss: 0.0061
Epoch [3/3], Loss: 0.0034
模型权重保存完成。

--- 开始导出适配 STM32N6 NPU 的优化静态 ONNX ---
融合失败，可能是模块名不匹配，跳过（不影响导出）
[torch.onnx] Obtain model graph for `TinyYoloFaceNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `TinyYoloFaceNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
Applied 12 of general pattern rewrite rules.
[torch.onnx] Optimize the ONNX graph... ✅
单文件静态 ONNX 已生成


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [ ]:
import os
import sys
import numpy as np

# ==========================================
# 0. 自动环境检查与依赖安装
# ==========================================
try:
    import onnx2tf
except ImportError:
    print(">>> 检测到当前环境未安装 onnx2tf，正在为您初始化边缘端转换环境...")
    !pip install -q onnx2tf onnx onnxsim tflite-support tensorflow
    print(">>> 依赖安装完毕！继续执行转换流程...\n")

print(">>> 正在全盘检索您的 ONNX 模型文件位置...")

# 1. 自动寻找由于重启可能发生路径位移的 onnx 文件
target_onnx = None
for root, dirs, files in os.walk("/content"):
    if "tiny_yolo_face_224_static.onnx" in files:
        target_onnx = os.path.join(root, "tiny_yolo_face_224_static.onnx")
        break

if target_onnx is None:
    for root, dirs, files in os.walk("/content"):
        for f in files:
            if "static.onnx" in f and f.endswith(".onnx"):
                target_onnx = os.path.join(root, f)
                break

if target_onnx is not None:
    print(f"【成功找到模型】绝对路径为: {target_onnx}")

    # 2. 重新构建 224x224 的校准矩阵
    work_dir = os.path.dirname(target_onnx)
    calib_path = os.path.join(work_dir, "calib_data.npy")

    print(">>> 正在生成 224x224 图像校准矩阵...")
    # 生成 100 张符合输入规格的随机样本用于激活值量化校准
    calib_data = np.random.rand(100, 224, 224, 3).astype(np.float32)
    np.save(calib_path, calib_data)

    print("\n>>> 开始调用锁定绝对路径的短标志命令进行重构 (ONNX -> TFLite INT8)...")
    # 3. 使用严格对齐新版本命令行的参数
    # !onnx2tf -i {target_onnx} \
    #      -oiqt \
    #      -ett full_integer_quant \
    #      -iqd uint8 \
    #      -oqd uint8 \
    #      -cind "Input_0" "{calib_path}"
    !onnx2tf -i {target_onnx} \
         -oiqt \
         -ett full_integer_quant \
         -iqd int8 \
         -oqd int8 \
         -qt per-tensor \
         -cind "Input_0" "{calib_path}"

    print("\n>>> 正在验证转换结果并进行软链接同步...")
    # 4. 提取生成的标准 int8 模型
    saved_model_dir = os.path.join(work_dir, "saved_model")
    target_output = "/content/stm32_face_yolo_int8.tflite"

    # 清理旧的模型缓存
    if os.path.exists(target_output):
        os.remove(target_output)

    # 自动检索生成的 int8/quant 文件
    found_int8 = False
    if os.path.exists(saved_model_dir):
        for f in os.listdir(saved_model_dir):
            # 匹配包含 'quant' 或 'int8' 关键字的 tflite 文件
            if ("quant" in f or "int8" in f) and f.endswith(".tflite"):
                src_path = os.path.join(saved_model_dir, f)
                !cp {src_path} {target_output}
                print(f"\n【大功告成】已成功捕获并导出 INT8 量化模型: {target_output}")
                print(f"原生成文件名: {f}")
                found_int8 = True
                break

    if not found_int8:
        print("\n❌【提示】未检测到自动生成的 int8 文件，请查看上方日志是否有量化报错。")
        if os.path.exists(saved_model_dir):
            print(f"当前 saved_model 目录下的文件有: {os.listdir(saved_model_dir)}")
else:
    print("\n❌【致命错误】未在当前 Colab 磁盘中找到任何 *.onnx 模型文件！")

>>> 正在全盘检索您的 ONNX 模型文件位置...
【成功找到模型】绝对路径为: /content/darknet/tiny_yolo_face_224_static.onnx
>>> 正在生成 224x224 图像校准矩阵...

>>> 开始调用锁定绝对路径的短标志命令进行重构 (ONNX -> TFLite INT8)...

Model optimizing started ============================================================
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Constant   │ 14             │ 14               │
│ Conv       │ 7              │ 7                │
│ LeakyRelu  │ 6              │ 6                │
│ MaxPool    │ 5              │ 5                │
│ Model Size │ 6.1MiB         │ 6.1MiB           │
└────────────┴────────────────┴──────────────────┘

Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Constant   │ 14             │ 14   

In [ ]:
import numpy as np
import cv2
import random

# 参数
NUM_SAMPLES = 300
IMG_SIZE = 224
TRAIN_LIST = 'face_train.txt'
SAVE_PATH = '/content/calib_data.npz'

# 读取清单
with open(TRAIN_LIST, 'r') as f:
    img_paths = [line.strip() for line in f if line.strip()]

print(f"共 {len(img_paths)} 张训练图片")
if len(img_paths) > NUM_SAMPLES:
    img_paths = random.sample(img_paths, NUM_SAMPLES)

calib_list = []
for path in img_paths:
    try:
        img = cv2.imread(path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img.astype(np.float32) / 255.0
        # 关键：从 HWC 转为 CHW（NCHW 的单张）
        img = img.transpose(2, 0, 1)   # (3, 224, 224)
        calib_list.append(img)
    except Exception as e:
        print(f"跳过 {path}，错误: {e}")

if not calib_list:
    raise RuntimeError("无有效图片！")

# 堆叠为 (N, 3, 224, 224)
calib_data = np.stack(calib_list, axis=0)
print(f"校准数据形状: {calib_data.shape}")   # 应该输出 (N, 3, 224, 224)

# 保存为 npz（键名为 calibration，可自定义）
np.savez_compressed(SAVE_PATH, calibration=calib_data)
print(f"已保存 {SAVE_PATH}")

共 1159 张训练图片
校准数据形状: (300, 3, 224, 224)
已保存 /content/calib_data.npz
